In [ ]:
# In the name of GOD Most Gracious Most Merciful

In [ ]:
# pip install scikit-learn==1.3.1 # the required version

In [ ]:
# pip install xgboost==2.0.3 # the required version

In [ ]:
# pip install tensorflow==2.13.0 # the required version 

In [ ]:
# Python == 3.11.4

In [ ]:
import numpy as np
import pandas as pd
import random
import h5py
import joblib
import time
import itertools

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
from tensorflow.compat.v1.losses import sparse_softmax_cross_entropy

from collections import deque 

import csv
import json
from json import JSONEncoder
from statistics import mode

from scipy.spatial.distance import cdist

In [ ]:
############################ DQNAgent class ##########################################
class DQNAgent:
    def __init__(self, state_dim, action_dim, batch_size, gamma, epsilon, epsilon_min, epsilon_decay, epsilon_initial, tau, dueling, ddqn):
        # Environment parameters
        self.state_dim = state_dim
        self.action_dim = action_dim

        # DQN alternatives
        self.ddqn = ddqn
        self.dueling = dueling
        
        # Neural network models
        self.model = self.build_model()
        self.target_model = self.build_model()

        
        # initialise target_model's weights with model's weights
        self.target_model.set_weights(self.model.get_weights())
        
        # Hyperparameters
        self.batch_size = batch_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.epsilon_initial = epsilon_initial
        self.tau = tau

        

    def build_model(self):
        X_input = Input(shape=(self.state_dim,))
        X = Dense(64, input_shape= (self.state_dim,), activation="relu")(X_input)
        X = Dense(64, activation='relu')(X)
        X = Dense(32, activation='relu')(X)
    
        if self.dueling:
            state_value = Dense(1)(X)
            action_advantage = Dense(self.action_dim)(X)
            X = (state_value + (action_advantage - tf.math.reduce_mean(action_advantage, axis=1, keepdims=True)))
        else:
            X = Dense(self.action_dim, activation="linear")(X)
            
    
        model = Model(inputs = X_input, outputs = X)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
        return model
    
    def get_action(self, state):
        mu = np.random.uniform(0,1) 
        
        if mu <= self.epsilon: 
            return np.random.choice([0, 1])
        else:
            input = np.array(state)  # Convert to a numpy array
            if input.ndim == 1:  
                input = np.expand_dims(input, axis=0)  # Reshape the input to a 2D array: (1, state_dim)
            q_values = self.model.predict(input)
            
            
            return np.argmax(q_values[0])
    
    
    # A method to set the replay buffer
    def set_replay_buffer(self, replay_buffer):  
        self.replay_buffer = replay_buffer
    
        
    # Train the deep learning model with a sample from replayBuffer
    def trainNetwork(self):
        optimal_Q_values = []
        epoch_losses =[]
        
        if len(self.replay_buffer) < self.batch_size:
            return 0, 0, 0 

        # sampling minibatch from replay buffer
        minibatch = self.replay_buffer.sample(self.batch_size)
        states, actions, rewards, next_states, dones = minibatch
        
        # Convert states into a list of DQN inputs
        dqn_inputs = [state for state in states]
        
        # Convert next_states into a list of DQN inputs
        next_dqn_inputs = [next_state for next_state in next_states]

        # Convert states and next_states into 2D numpy arrays
        dqn_inputs = np.array(dqn_inputs)  # Shape: (batch_size, state_dim)
        next_dqn_inputs = np.array(next_dqn_inputs)  # Shape: (batch_size, state_dim)


        
        target = self.model.predict(dqn_inputs)
        target_next = self.model.predict(next_dqn_inputs)
        target_val = self.target_model.predict(next_dqn_inputs)

        

        for i in range(self.batch_size):
            if dones[i]:
                target[i][actions[i]] = rewards[i]    
            else:
                if self.ddqn: # Double - DQN  # Choose DQN or Double DQN using self.ddqn
                    a = np.argmax(target_next[i]) 
                    target[i][actions[i]] = rewards[i] + self.gamma * target_val[i][a] 
                else: # Standard - DQN
                    target[i][action[i]] = reward[i] + self.gamma * (np.amax(target_next[i]))

            optimal_Q_values.append(target[i][actions[i]])
            
        mean_q_value = np.mean(optimal_Q_values)
        max_q_value = np.max(optimal_Q_values)
        

        history = self.model.fit(dqn_inputs, target, epochs=50, verbose=0)
        epoch_losses.extend(history.history['loss'])

        return mean_q_value, max_q_value, epoch_losses
        
    def Q_value_calculate(self, state, action, reward, next_state, done):
        # Convert states into a list of DQN inputs
        dqn_input = np.array(state)
        dqn_input = np.expand_dims(dqn_input, axis=0)
        
        # Convert next_states into a list of DQN inputs
        next_dqn_input = np.array(next_state)
        next_dqn_input = np.expand_dims(next_dqn_input, axis=0)


        
        target = self.model.predict(dqn_input)
        target_next = self.model.predict(next_dqn_input)
        target_val = self.target_model.predict(next_dqn_input)
        
        
        if done:
            Q = reward   
        else:
            if self.ddqn: # Double - DQN
                a = np.argmax(target_next[0]) 
                Q = reward + self.gamma * target_val[0][a] 
            else: # Standard - DQN
                Q = reward + self.gamma * (np.amax(target_next[0]))

        return Q
        
        
        
    
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def update_target_model_soft(self):
        target_model_weights = self.target_model.get_weights()
        model_weights = self.model.get_weights()
        new_weights = []
        for target_model_weight, model_weight in zip(target_model_weights, model_weights):
            new_weight = self.tau * model_weight + (1 - self.tau) * target_model_weight
            new_weights.append(new_weight)
        self.target_model.set_weights(new_weights)
        

    def update_epsilon(self, total_time):
        if self.epsilon > self.epsilon_min:
            self.epsilon = self.epsilon_initial / (1+total_time*self.epsilon_decay)

In [ ]:
################# Replay buffer class #########################################
class ReplayBuffer:
    def __init__(self, max_size):
        self.max_size = max_size
        self.buffer = deque(maxlen=max_size)

    def push(self, experience):
        self.buffer.append(experience)
        
    def __len__(self):
        return len(self.buffer)

    def sample(self, batch_size):
        if len(self.buffer) < batch_size:
            batch_size = len(self.buffer)

        batch = random.sample(self.buffer, batch_size)
        state_batch, action_batch, reward_batch, next_state_batch, done_batch = [], [], [], [], []

        for experience in batch:
            state, action, reward, next_state, done = experience
            state_batch.append(state)
            action_batch.append(action)
            reward_batch.append(reward)
            next_state_batch.append(next_state)
            done_batch.append(done)

        return state_batch, action_batch, reward_batch, next_state_batch, done_batch

In [ ]:
def Reward(alert_DQN):
    if alert_DQN['priority_Ensemble'] == alert_DQN['revised_priority']:
        return -5  # Negative reward if the priorities match
    else:
        # Positive rewards based on the revised priority
        if alert_DQN['revised_priority'] == 4:
            return 9
        elif alert_DQN['revised_priority'] == 3:
            return 7
        elif alert_DQN['revised_priority'] == 2:
            return 5
        elif alert_DQN['revised_priority'] == 1:
            return 3
        elif alert_DQN['revised_priority'] == 0:
            return 1

In [ ]:
# A function to add alert to AVAR (analyst-validated alert repository)
def add_alert_to_storage(alert_DQN):
    columns = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12']
    
    # Create a new DataFrame row from alert['Attribute']
    new_row = pd.DataFrame([alert_DQN['Attribute']], columns=columns)
    
    # Define the mapping of priority to file paths
    storage_files = {
        0: 'Normal_storage.csv',
        1: 'Low_storage.csv',
        2: 'Moderate_storage.csv',
        3: 'High_storage.csv',
        4: 'Critical_storage.csv'
    }
    
    # Get the storage file of AVAR based on the revised priority
    storage_file = storage_files.get(alert_DQN['revised_priority'])

    # print('storage_file =', storage_file)
    
    if storage_file:
        # Append the new row directly to the file without a header
        new_row.to_csv(storage_file, index=False, header=False, mode='a')


In [ ]:
    
def step(alert_DQN, state, action, step_length, index, investigation_time_alert): 
                                                                       
                                                                      
    if action == 1: # present alert to analyst
        alert_DQN['revised_priority'] = alert_DQN['priority_Analyst']  # this is the presentation to the analyst
       
        
        alert_DQN['allocated_time'] = investigation_time_alert
        
    
        reward = Reward(alert_DQN)
        
        # next state
        alert_DQN['alert_state'][-1] = alert_DQN['revised_priority']
        next_state = alert_DQN['alert_state']
         
        # update Storages (AVAR)
        add_alert_to_storage(alert_DQN) 
                                                                                                             
        
    elif action == 0: # Not present alert to analyst
        
        
        alert_DQN['allocated_time'] = 0
        
        reward = 0  # alert_DQN['revised_priority'] remains None since the alert is not presented to the analyst 
        
        # next state
        next_state = alert_DQN['alert_state']  
        
    
    if index < step_length - 1:
        return alert_DQN, reward, next_state, False
    else:
        return alert_DQN, reward, next_state, True

In [ ]:
def AP(agent, replay_buffer, num_episodes, num_time_steps, Features, CVSS_df, combined_alert_data, investigation_times):


    # Initialise an empty dictionary to store metrics
    metrics_data = {}
    size_alert_left = len(Features)
    lower_bound = 0

    # Load classifer models (for Ensemble model)
    rf_clf = joblib.load('RF_whole_model_CIC_mapped.joblib')
    adb_clf = joblib.load('ADB_whole_model_CIC_mapped.joblib')
    xgb_clf = joblib.load('XGB_whole_model_CIC_mapped.joblib')
    nb_clf = joblib.load('NB_whole_model_CIC_mapped.joblib')
    
    for episode in range(num_episodes):

        episode_key = f"episode_{episode + 1}"  # Use a dynamic key for each episode
        
        # Initialise an empty dictionary for the current episode metrics
        episode_metrics = {}

        # Poisson distribution
        # Mean number of alerts per hour (λ)
        lambda_ = 400
        
        # Number of time steps (hours)
        steps = num_time_steps  # 24
        
        # Generate the number of alerts for each time step using Poisson distribution
        alerts_number_all_time_steps = np.random.poisson(lambda_, steps)
        
        
        for time_step in range(num_time_steps):
            
            start_time = time.time()  # Record the start time
            
            time_step_key = f"time_step_{time_step + 1}"  # Use a dynamic key for each time step
            
            # for my clarification
            time_step = time_step 

            # The total time of run so far
            total_time = (episode * num_time_steps) + time_step

            ########## Alert selection for each time_step ######################################################################

            # The number of input alerts to SOC based on Poisson distribution  
            alerts_per_step = alerts_number_all_time_steps[time_step]
        
            # Determine how many alerts to select
            alerts_to_select = min(alerts_per_step, size_alert_left)

            
            upper_bound = lower_bound + alerts_to_select
            
            
            # Select the next batch of alerts
            selected_alert_data = Features.iloc[lower_bound : upper_bound]
            
        
            # Select the reprioritisation values of alerts      
            selected_reprioritisation_data = CVSS_df[lower_bound : upper_bound]
            

            selected_combined_alert_data = combined_alert_data[lower_bound : upper_bound]
            
            
            # alerts for each time_step
            size_alert_left = size_alert_left - len(selected_alert_data)
            lower_bound = upper_bound
            
            ###############################################################################################################                              
            ################ Ensemble prioritisation ######################################################################

            X_new = selected_alert_data.to_numpy()
            

            CVSS_label = selected_reprioritisation_data['CVSS_value']            # Can be used for the analyst
            Identifiers = selected_reprioritisation_data['Identifier'].tolist()
            
            Attack_types = selected_reprioritisation_data[' Label'].tolist()
            
            
            # Make prioritisation using each individual model
            rf_predictions_mapped = rf_clf.predict(X_new)
            adb_predictions_mapped = adb_clf.predict(X_new)
            xgb_predictions_mapped = xgb_clf.predict(X_new)
            nb_predictions_mapped = nb_clf.predict(X_new)
            
            
            # Map and De-map y values to/from sequential labels
            # Map
            label_mapping = {0: 0, 2: 1, 3: 2, 4: 3}
            #De-map
            reverse_mapping = {v: k for k, v in label_mapping.items()}

            # predictions with real categories
            rf_predictions = np.vectorize(reverse_mapping.get)(rf_predictions_mapped)
            adb_predictions = np.vectorize(reverse_mapping.get)(adb_predictions_mapped)
            xgb_predictions = np.vectorize(reverse_mapping.get)(xgb_predictions_mapped)
            nb_predictions = np.vectorize(reverse_mapping.get)(nb_predictions_mapped)

            
            
            # Combine the four prediction lists into a single list of tuples
            combined_predictions = zip(rf_predictions, adb_predictions, xgb_predictions, nb_predictions)
        
            # Create a list of lists, where each inner list contains the corresponding values from the four lists
            all_predictions = [list(prediction_tuple) for prediction_tuple in combined_predictions]
        
            all_predictions = np.array(all_predictions)
            
            
            modes = []
            for values in all_predictions:
                mode_value = mode(values)
                modes.append(mode_value)
            
        
            # Choose the predictions from all models using majority voting
            ensemble_prioritisation = modes
            
        
            # Ensure ensemble_prioritisation are numpy arrays
            ensemble_prioritisation = np.array(ensemble_prioritisation)
                

            ##############################################################################################################
            ######## Building the state dimensions for each alert #########

            alert_list =[]
            # Convert predicted_alerts (prioritised alerts) into a list of dictionaries with required features
            for index, value in enumerate(ensemble_prioritisation):
                priority = ''
                if value == 4:
                    priority = 'Critical'
                elif value == 3:
                    priority = 'High'
                elif value == 2:
                    priority = 'Moderate'
                elif value == 1:
                    priority = 'Low'
                else:
                    priority = 'Normal'

                
                alert_id = f'{total_time}_{index+1}'
                

                # Create the dictionary for each alert
                alert_dict = {
                    'alert_id': alert_id,
                    'Identifier': Identifiers[index],
                    'Attack_type': Attack_types[index],
                    'priority_Ensemble': value,     
                    'priority_Analyst': CVSS_label.iloc[index],
                    'Attribute': selected_alert_data.iloc[index].tolist(),
                    'P1': None,
                    'P2': None,
                    'P3': None,
                    'P4': None,
                    'P5': None,
                    'P6': None,
                    'Minimum_Distance': None,
                    'Old_alert': None,
                    'revised_priority': 10,
                    'allocated_time': None,
                    'alert_state': None,
                    'alert_action': None,
                    'alert_reward': None
                }

                

                # Append the dictionary to the list
                alert_list.append(alert_dict)
                

            
            # Here, we include Normal alerts as well. # Since accuracy Normal < 1, Normal is not removed
            # But, to avoid changing the name of variables and the code, we use the name 'alert_list_without_Normal'
            alert_list_without_Normal = alert_list 

            
            # Reading storages of AVAR from .csv files to DataFrames
            Critical_storage = pd.read_csv('Critical_storage.csv')
            High_storage = pd.read_csv('High_storage.csv')
            Moderate_storage = pd.read_csv('Moderate_storage.csv')
            Low_storage = pd.read_csv('Low_storage.csv')
            Normal_storage = pd.read_csv('Normal_storage.csv')

            
            
            # lists to collect alerts whose 'revised_priority' are assigned and are not assigned
            alert_list_without_Normal_with_revised_priority = []
            alert_list_without_Normal_without_revised_priority = []

            
            # Function to calculate Euclidean distance
            def mean_euclidean_distance(new_alert, storage_df):
                # Ensure new_alert is a NumPy array with the correct shape
                new_alert = np.array(new_alert).reshape(1, -1)
                # Convert the storage DataFrame to a NumPy array for computation
                storage_array = storage_df.values
                # Compute pairwise distances between new_alert and all rows in storage_array
                distances = cdist(new_alert, storage_array, metric='euclidean')
                # Calculate and return the mean of the distances
                mean_distance = np.mean(distances)
                return mean_distance
                

            for alert in alert_list_without_Normal:
                
                # Check whether it is an old alert in AVAR storages
                # Extract the 'Attribute' list from the alert_dict
                alert_features = alert['Attribute']

                # Column names for the PCA features
                pca_columns = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12']

                
                # Check for old alerts
                old_alert_Critical = Critical_storage[(Critical_storage[pca_columns].values == alert_features).all(axis=1)]
                old_alert_High = High_storage[(High_storage[pca_columns].values == alert_features).all(axis=1)]
                old_alert_Moderate = Moderate_storage[(Moderate_storage[pca_columns].values == alert_features).all(axis=1)]
                old_alert_Low = Low_storage[(Low_storage[pca_columns].values == alert_features).all(axis=1)]
                old_alert_Normal = Normal_storage[(Normal_storage[pca_columns].values == alert_features).all(axis=1)]

                olds = {
                    4: len(old_alert_Critical),
                    3: len(old_alert_High),
                    2: len(old_alert_Moderate),
                    1: len(old_alert_Low),
                    0: len(old_alert_Normal)
                }

                # Check conditions
                all_zero = all(value == 0 for value in olds.values())
                more_than_two_non_zero = sum(1 for value in olds.values() if value > 0) >= 2
                
                if all_zero or more_than_two_non_zero:
                    result_old = False
                    priority_old = None
                else:
                    result_old = True
                    priority_old = max(key for key, value in olds.items() if value > 0)
                    alert['revised_priority'] = priority_old

                #@@@@@@@@@@@@@@@@@@@@@@@@
                alert['Old_alert'] = result_old
                #@@@@@@@@@@@@@@@@@@@@@@@

                if alert['revised_priority'] != 10:
                    alert_list_without_Normal_with_revised_priority.append(alert)
                else: # Build the alert state for DQN analysis
                
                    alert_attributes = np.array(alert['Attribute'])
    
                    # Euclidean Distance to storages
                    Critical_mahal = mean_euclidean_distance(alert_attributes, Critical_storage)
                    High_mahal = mean_euclidean_distance(alert_attributes, High_storage)
                    Moderate_mahal = mean_euclidean_distance(alert_attributes, Moderate_storage)
                    Low_mahal = mean_euclidean_distance(alert_attributes, Low_storage)
                    Normal_mahal = mean_euclidean_distance(alert_attributes, Normal_storage)
    
                    # Find the minimum distance and corresponding category
                    distances = {
                        4: Critical_mahal,
                        3: High_mahal,
                        2: Moderate_mahal,
                        1: Low_mahal,
                        0: Normal_mahal
                    }
    
                    min_value = min(distances.values())
                    # Find all keys with the min value
                    result_distances = [key for key, value in distances.items() if value == min_value]
                    
    
                    if len(result_distances) == 1:
                        min_category = result_distances[0]
                    else:
                        min_category = 10  # when min_distance is linked to more than one category and cannot be identified  
                    
    
    
                    #@@@@@@@@ (1) State distance @@@@@@@@@@@@@@@@
                    state_distance = min_category 
                    alert['Minimum_Distance'] = state_distance
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    
                    
                    # Check the P features of the alert belong to which AVAR storages
    
                    # Check the P features in Critical_storage
                    
                    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
                    P1_in_Critical_storage = alert_features[0] in Critical_storage['PC1'].values
                    
                    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
                    P2_in_Critical_storage = alert_features[1] in Critical_storage['PC2'].values
                    
                    # Check if the third PCA feature of the new alert is among the third PCA feature column of df
                    P3_in_Critical_storage = alert_features[2] in Critical_storage['PC3'].values
    
                    # Check if the fourth PCA feature of the new alert is among the fourth PCA feature column of df
                    P4_in_Critical_storage = alert_features[3] in Critical_storage['PC4'].values
                    
                    # Check if the fifth PCA feature of the new alert is among the fifth PCA feature column of df
                    P5_in_Critical_storage = alert_features[4] in Critical_storage['PC5'].values
                    
                    # Check if the sixth PCA feature of the new alert is among the sixth PCA feature column of df
                    P6_in_Critical_storage = alert_features[5] in Critical_storage['PC6'].values
                    
                    
    
                    # Check the P features in High_storage
                    
                    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
                    P1_in_High_storage = alert_features[0] in High_storage['PC1'].values
                    
                    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
                    P2_in_High_storage = alert_features[1] in High_storage['PC2'].values
                    
                    # Check if the third PCA feature of the new alert is among the third PCA feature column of df
                    P3_in_High_storage = alert_features[2] in High_storage['PC3'].values
    
                    # Check if the fourth PCA feature of the new alert is among the fourth PCA feature column of df
                    P4_in_High_storage = alert_features[3] in High_storage['PC4'].values
                    
                    # Check if the fifth PCA feature of the new alert is among the fifth PCA feature column of df
                    P5_in_High_storage = alert_features[4] in High_storage['PC5'].values
                    
                    # Check if the sixth PCA feature of the new alert is among the sixth PCA feature column of df
                    P6_in_High_storage = alert_features[5] in High_storage['PC6'].values
                    
                    
    
                    # Check the P features in Moderate_storage
                    
                    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
                    P1_in_Moderate_storage = alert_features[0] in Moderate_storage['PC1'].values
                    
                    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
                    P2_in_Moderate_storage = alert_features[1] in Moderate_storage['PC2'].values
                    
                    # Check if the third PCA feature of the new alert is among the third PCA feature column of df
                    P3_in_Moderate_storage = alert_features[2] in Moderate_storage['PC3'].values
    
                    # Check if the fourth PCA feature of the new alert is among the fourth PCA feature column of df
                    P4_in_Moderate_storage = alert_features[3] in Moderate_storage['PC4'].values
                    
                    # Check if the fifth PCA feature of the new alert is among the fifth PCA feature column of df
                    P5_in_Moderate_storage = alert_features[4] in Moderate_storage['PC5'].values
                    
                    # Check if the sixth PCA feature of the new alert is among the sixth PCA feature column of df
                    P6_in_Moderate_storage = alert_features[5] in Moderate_storage['PC6'].values
                    
                    
    
                    # Check the P features in Low_storage
                    
                    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
                    P1_in_Low_storage = alert_features[0] in Low_storage['PC1'].values
                    
                    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
                    P2_in_Low_storage = alert_features[1] in Low_storage['PC2'].values
                    
                    # Check if the third PCA feature of the new alert is among the third PCA feature column of df
                    P3_in_Low_storage = alert_features[2] in Low_storage['PC3'].values
    
                    # Check if the fourth PCA feature of the new alert is among the fourth PCA feature column of df
                    P4_in_Low_storage = alert_features[3] in Low_storage['PC4'].values
                    
                    # Check if the fifth PCA feature of the new alert is among the fifth PCA feature column of df
                    P5_in_Low_storage = alert_features[4] in Low_storage['PC5'].values
                    
                    # Check if the sixth PCA feature of the new alert is among the sixth PCA feature column of df
                    P6_in_Low_storage = alert_features[5] in Low_storage['PC6'].values
                    
    
                    # Check the P features in Normal_storage
                    
                    # Check if the first PCA feature of the new alert is among the first PCA feature column of df
                    P1_in_Normal_storage = alert_features[0] in Normal_storage['PC1'].values
                    
                    # Check if the second PCA feature of the new alert is among the second PCA feature column of df
                    P2_in_Normal_storage = alert_features[1] in Normal_storage['PC2'].values
                    
                    # Check if the third PCA feature of the new alert is among the third PCA feature column of df
                    P3_in_Normal_storage = alert_features[2] in Normal_storage['PC3'].values
    
                    # Check if the fourth PCA feature of the new alert is among the fourth PCA feature column of df
                    P4_in_Normal_storage = alert_features[3] in Normal_storage['PC4'].values
                    
                    # Check if the fifth PCA feature of the new alert is among the fifth PCA feature column of df
                    P5_in_Normal_storage = alert_features[4] in Normal_storage['PC5'].values
                    
                    # Check if the sixth PCA feature of the new alert is among the sixth PCA feature column of df
                    P6_in_Normal_storage = alert_features[5] in Normal_storage['PC6'].values
                    
                    
                    #set the state items related to alert features
                    P1 = {
                        4: P1_in_Critical_storage,
                        3: P1_in_High_storage,
                        2: P1_in_Moderate_storage,
                        1: P1_in_Low_storage,
                        0: P1_in_Normal_storage
                    }
    
    
                    # Check conditions
                    all_false = all(value == False for value in P1.values())
                    more_than_two_true = sum(1 for value in P1.values() if value == True) >= 2
                    
                    if all_false or more_than_two_true:
                        result_P1 = 10
                    else:
                        result_P1 = max(key for key, value in P1.items() if value == True)
                    
                        
                    #@@@@@ (2) State P1 @@@@@@@
                    state_P1 = result_P1
                    alert['P1'] = state_P1
                    #@@@@@@@@@@@@@@@@@@@@@@@
    
                    P2 = {
                        4: P2_in_Critical_storage,
                        3: P2_in_High_storage,
                        2: P2_in_Moderate_storage,
                        1: P2_in_Low_storage,
                        0: P2_in_Normal_storage
                    }
    
                    # Check conditions
                    all_false = all(value == False for value in P2.values())
                    more_than_two_true = sum(1 for value in P2.values() if value == True) >= 2
                    
                    if all_false or more_than_two_true:
                        result_P2 = 10
                    else:
                        result_P2 = max(key for key, value in P2.items() if value == True)
                    
    
                    #@@@@@ (3) State P2 @@@@@@@
                    state_P2 = result_P2
                    alert['P2'] = state_P2
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@
    
    
                    P3 = {
                        4: P3_in_Critical_storage,
                        3: P3_in_High_storage,
                        2: P3_in_Moderate_storage,
                        1: P3_in_Low_storage,
                        0: P3_in_Normal_storage
                    }
    
                    # Check conditions
                    all_false = all(value == False for value in P3.values())
                    more_than_two_true = sum(1 for value in P3.values() if value == True) >= 2
                    
                    if all_false or more_than_two_true:
                        result_P3 = 10
                    else:
                        result_P3 = max(key for key, value in P3.items() if value == True)
    
                    #@@@@@ (4) State P3 @@@@@@@
                    state_P3 = result_P3
                    alert['P3'] = state_P3
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@
                    
    
                    P4 = {
                        4: P4_in_Critical_storage,
                        3: P4_in_High_storage,
                        2: P4_in_Moderate_storage,
                        1: P4_in_Low_storage,
                        0: P4_in_Normal_storage
                    }
    
                    # Check conditions
                    all_false = all(value == False for value in P4.values())
                    more_than_two_true = sum(1 for value in P4.values() if value == True) >= 2
                    
                    if all_false or more_than_two_true:
                        result_P4 = 10
                    else:
                        result_P4 = max(key for key, value in P4.items() if value == True)
    
                    #@@@@@ (5) State P4 @@@@@@@
                    state_P4 = result_P4
                    alert['P4'] = state_P4
                    #print(' alert[P4] =',  alert['P4'])
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@
    
                    P5 = {
                        4: P5_in_Critical_storage,
                        3: P5_in_High_storage,
                        2: P5_in_Moderate_storage,
                        1: P5_in_Low_storage,
                        0: P5_in_Normal_storage
                    }
    
                    # Check conditions
                    all_false = all(value == False for value in P5.values())
                    more_than_two_true = sum(1 for value in P5.values() if value == True) >= 2
                    
                    if all_false or more_than_two_true:
                        result_P5 = 10
                    else:
                        result_P5 = max(key for key, value in P5.items() if value == True)
    
                    #@@@@@ (6) State P5 @@@@@@@
                    state_P5 = result_P5
                    alert['P5'] = state_P5
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@
    
                    P6 = {
                        4: P6_in_Critical_storage,
                        3: P6_in_High_storage,
                        2: P6_in_Moderate_storage,
                        1: P6_in_Low_storage,
                        0: P6_in_Normal_storage
                    }
    
                    # Check conditions
                    all_false = all(value == False for value in P6.values())
                    more_than_two_true = sum(1 for value in P6.values() if value == True) >= 2
                    
                    if all_false or more_than_two_true:
                        result_P6 = 10
                    else:
                        result_P6 = max(key for key, value in P6.items() if value == True)
    
                    #@@@@@ (7) State P6 @@@@@@@
                    state_P6 = result_P6
                    alert['P6'] = state_P6
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@
    
    
                    #@@@@@@@@ alert state @@@@@@@@@@@@@@@@@@@@@
                    alert_state = [alert['priority_Ensemble'], state_P1, state_P2, state_P3, state_P4, state_P5, state_P6, state_distance, alert['revised_priority']]
                    alert['alert_state'] = alert_state
                    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    
                    # Add the alert to the list of without revised priority to send to DQN
                    alert_list_without_Normal_without_revised_priority.append(alert)
                
 ################# Interacting DQN with analyst ########################   

            
            ## Sort alerts based on their Ensemble priority to present to DQN
            sorted_alert_list_without_Normal_without_revised_priority = sorted(alert_list_without_Normal_without_revised_priority, key=lambda alert: alert['priority_Ensemble'], reverse=True)
            
            step_length = len(sorted_alert_list_without_Normal_without_revised_priority)
            analyst_time_percentage_for_analysis = 0.8
            remaining_time = int(60 * 60 * analyst_time_percentage_for_analysis) # analyst available time to review alerts in a time step (in seconds)
            

            # Lists of performance
            alerts_after_action_015 = []
            alerts_after_action_01 = [] # this gathers alerts after DQN decision (0: accept, 1: defer)
            reward_list = []
            Q_value_each_state_list = []
            mean_q_value_list = []
            max_q_value_list = []
            epoch_losses_list = []

            def calculate_average(list_A):
                if len(list_A) == 0:
                    return 0
                return sum(list_A) / len(list_A)
            
            while remaining_time > 0:
                for index, alert_DQN in enumerate(sorted_alert_list_without_Normal_without_revised_priority):
                    
                    # get investigation time
                    Ensemble_priority = alert_DQN['priority_Ensemble']
                    
                    investigation_time_alert = investigation_times.get(Ensemble_priority)
                    
                    if remaining_time < investigation_time_alert:
                        action = 5  # It means that alert is not presented because of analyst time constraint 
                        alert_DQN['alert_action'] = action
                        
                    else: # DQN starts 
                        # state 
                        state = alert_DQN['alert_state']
                        
                        # action
                        action = agent.get_action(state)
                        
                        # step
                        alert_DQN, reward, next_state, done = step(alert_DQN, state, action, step_length, index, investigation_time_alert) 
                                                                                                                                                  
                                                                                                                                                  
                        
                                                                   
                        
                        alert_DQN['alert_action'] = action
                        alert_DQN['alert_reward'] = reward
                        
                        
                        
                         
                        
                        # Store the experience in the replay buffer
                        replay_buffer.push((state, action, reward, next_state, done))
                        
                        # train the DQN network
                        mean_q_value, max_q_value, epoch_losses = agent.trainNetwork()

                        
                        #soft update the target network
                        agent.update_target_model_soft()

                        # calculate Q value of this experience (each state) (optional-not important)
                        Q_value = agent.Q_value_calculate(state, action, reward, next_state, done)

                        

            
                        # update performance lists 
                        reward_list.append(reward)
                        mean_q_value_list.append(mean_q_value)
                        max_q_value_list.append(max_q_value)
                        Q_value_each_state_list.append(Q_value)
                        epoch_losses_list.append(epoch_losses)
                        alerts_after_action_01.append(alert_DQN)

                        # update remaining time
                        remaining_time -= alert_DQN['allocated_time']
                        
                    
                if remaining_time > 0:
                    print(f"Loop terminated with remaining time: {remaining_time}")
                    break

                        
            end_time = time.time()  # Record the end time
            time_step_execution_time = end_time - start_time  # Calculate the duration of execution of this loop         
                    
            
            # Update the epsilon value for epsilon-greedy exploration each time_step
            agent.update_epsilon(total_time)
                    
            # all alerts without Normal of the time step after actions
            Final_alert_list_without_Normal_addressed = list(itertools.chain(alert_list_without_Normal_with_revised_priority, alerts_after_action_01))
            reward_average = calculate_average(reward_list) 
            mean_q_value_average = calculate_average(mean_q_value_list)
            max_q_value_average = calculate_average(max_q_value_list)
            Q_value_each_state_average = calculate_average(Q_value_each_state_list)
               

            # Extract alert_ids from alerts_after_action_01
            alert_ids_after_action_01 = {alert['alert_id'] for alert in alerts_after_action_01}
            
            
            # Un_investigated alerts by analyst
            Final_alert_list_without_Normal_NotAddressed = [
                alert for alert in alert_list_without_Normal_without_revised_priority
                if alert['alert_id'] not in alert_ids_after_action_01
            ]
            
            Analyst_Workload = len(Final_alert_list_without_Normal_NotAddressed)
            
            # Store metrics in the current time step
            current_metrics = {
                "Number_total_alerts": len(selected_alert_data),
                "Mean_Q_values": mean_q_value_average,
                "Max_Q_values": max_q_value_average,
                "Q_value_each_state_average": Q_value_each_state_average,
                "Q_value_each_state_list": Q_value_each_state_list,
                "Epoch_Losses": epoch_losses_list,
                "Reward": reward_average,
                "Reward_list": reward_list,
                "alert_list_without_Normal": alert_list_without_Normal,
                "alert_list_without_Normal_with_revised_priority": alert_list_without_Normal_with_revised_priority, 
                "alert_list_without_Normal_without_revised_priority": alert_list_without_Normal_without_revised_priority, 
                "alerts_after_action_01": alerts_after_action_01,
                "Final_alert_list_without_Normal_addressed": Final_alert_list_without_Normal_addressed,
                "Final_alert_list_without_Normal_NotAddressed": Final_alert_list_without_Normal_NotAddressed,
                "time_step_execution_time": time_step_execution_time,
                "Analyst_Workload": Analyst_Workload,
                "Remaining_time": remaining_time 
            }
            
            episode_metrics[time_step_key] = current_metrics
            
            class NumpyEncoder(json.JSONEncoder):
                def default(self, obj):
                    if isinstance(obj, np.ndarray):
                        return obj.tolist()
                    elif isinstance(obj, np.integer):
                        return int(obj)
                    elif isinstance(obj, np.floating):
                        return float(obj)
                    return super(NumpyEncoder, self).default(obj)

            
            output_folder = 'C:\\...\\L2DHF_CICIDS2017\\'
           
            
            # Save the dictionary of metrics to a JSON file
            with open(f'{output_folder}L2DHF_CICIDS2017_time_step_{total_time + 1}.json', 'w') as outfile:
                json.dump(current_metrics, outfile, cls=NumpyEncoder)

            
            

        # Store the episode metrics in the main dictionary
        metrics_data[episode_key] = episode_metrics
        
        
    return metrics_data

In [ ]:
############################################### Load the alert dataset ######################################################################
################ pca_12_one_hot
#### subset 1, 2, 3, 4, 5 
df_PCA_12_combined_data_CIC_reduced_normalised_1 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_1.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_2 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_2.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_3 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_3.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_4 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_4.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_5 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_5.csv')

In [ ]:
# Shuffle data
df_PCA_12_combined_data_CIC_reduced_normalised_1 = df_PCA_12_combined_data_CIC_reduced_normalised_1.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_2 = df_PCA_12_combined_data_CIC_reduced_normalised_2.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_3 = df_PCA_12_combined_data_CIC_reduced_normalised_3.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_4 = df_PCA_12_combined_data_CIC_reduced_normalised_4.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_5 = df_PCA_12_combined_data_CIC_reduced_normalised_5.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Combine 5 sub_datasets to a dataset
combined_alert_data = pd.concat([df_PCA_12_combined_data_CIC_reduced_normalised_1, df_PCA_12_combined_data_CIC_reduced_normalised_2, df_PCA_12_combined_data_CIC_reduced_normalised_3, df_PCA_12_combined_data_CIC_reduced_normalised_4, df_PCA_12_combined_data_CIC_reduced_normalised_5], ignore_index=True)

In [ ]:
# Shuffle combined_alert_data (second shuffle)
combined_alert_data = combined_alert_data.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_1

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_2

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_3

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_4

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_5

In [ ]:
combined_alert_data

In [ ]:
############# Building the AVAR Storages of categories #################

In [ ]:
# Number of CVSS categories in the dataset
cvss_counts = combined_alert_data['CVSS'].value_counts()
cvss_counts

In [ ]:
# Building the AVAR storages of Critical, High, Medium, Low, and Normal
# Set the desired sample sizes
sample_sizes = {
    'Critical': 5,
    'High': 5,
    'Moderate': 5,
    'Low': 0,
    'Normal': 5
}

# Initialise a dictionary to hold the sampled df
sampled_dfs = {}

# Initialise a list to collect the indices of sampled rows
sampled_indices = []

# Iterate over the sample sizes dictionary
for cvss_value, sample_size in sample_sizes.items():
    # Filter the df for the current CVSS value
    filtered_combined_alert_data = combined_alert_data[combined_alert_data['CVSS'] == cvss_value]
    
    # Randomly sample the specified number of rows
    sampled_df = filtered_combined_alert_data.sample(n=sample_size, random_state=42)
    
    
    sampled_dfs[cvss_value] = sampled_df
    
    
    sampled_indices.extend(sampled_df.index)

# Drop the sampled rows from combined_alert_data
combined_alert_data = combined_alert_data.drop(sampled_indices)

# Separate sets for each CVSS value 
critical_set = sampled_dfs['Critical']
high_set = sampled_dfs['High']
moderate_set = sampled_dfs['Moderate']
low_set = sampled_dfs['Low']
normal_set = sampled_dfs['Normal']


In [ ]:
# Save storages to CSV files
critical_set.to_csv('critical_set.csv', index=False)
high_set.to_csv('high_set.csv', index=False)
moderate_set.to_csv('moderate_set.csv', index=False)
#low_set.to_csv('low_set.csv', index=False) # Low category is removed because it does not exist in CICIDS2017 dataset
normal_set.to_csv('normal_set.csv', index=False)

In [ ]:
critical_set

In [ ]:
high_set

In [ ]:
moderate_set

In [ ]:
low_set

In [ ]:
# A fake rwo for low category is created to generate low storage and keep the code consistent with the codes related to UNSW-NB15 dataset
low_set.loc[0] = [10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, "BENIGN", "Normal", 0, "ID_0000000"]

In [ ]:
low_set

In [ ]:
normal_set

In [ ]:
# Remove labels form storages
columns_to_remove = [' Label', 'CVSS', 'CVSS_value', 'Identifier']
Critical_storage = critical_set.drop(columns=columns_to_remove, axis=1)
High_storage = high_set.drop(columns=columns_to_remove, axis=1)
Moderate_storage = moderate_set.drop(columns=columns_to_remove, axis=1) 
Low_storage = low_set.drop(columns=columns_to_remove, axis=1)
Normal_storage = normal_set.drop(columns=columns_to_remove, axis=1)

In [ ]:
print('Len Critical_storage =',len(Critical_storage))
print('Len High_storage =', len(High_storage))
print('Len Moderate_storage =', len(Moderate_storage))
print('Len Low_storage =', len(Low_storage))
print('Len Normal_storage =', len(Normal_storage))

In [ ]:
Critical_storage

In [ ]:
High_storage

In [ ]:
Moderate_storage

In [ ]:
Low_storage

In [ ]:
Normal_storage

In [ ]:
# Save df storages to .csv files
Critical_storage.to_csv('Critical_storage.csv', index=False)
High_storage.to_csv('High_storage.csv', index=False)
Moderate_storage.to_csv('Moderate_storage.csv', index=False)
Low_storage.to_csv('Low_storage.csv', index=False)
Normal_storage.to_csv('Normal_storage.csv', index=False)

In [ ]:
######### Building alerts without labels for prioritisation and separate labels ############

In [ ]:
# Separating alert features and re-prioritisation columns 
columns_to_remove = [' Label', 'CVSS', 'CVSS_value', 'Identifier']
Features = combined_alert_data.drop(columns=columns_to_remove, axis=1)
CVSS_df = combined_alert_data[['CVSS_value', ' Label', 'Identifier']]

In [ ]:
CVSS_df

In [ ]:
Features

In [ ]:
############################################## Main program ###########################################
if __name__ == "__main__":
    # state and action size
    state_dim = 9    
    action_dim = 2  
      
    # time parameters
    num_episodes = 7 * 12  # days of SOC working 12*7=84
    num_time_steps = 24  # hours of day for SOC working, # one working shift  
    
    # investigation times of analyst for alert categories (in seconds)
    investigation_times = {
        4: 270,   
        3: 210,   
        2: 120,   
        1: 90,    
        0: 60    
    }
    
    # Hyperparameters
    max_size = 1000 # size of replay_buffer 
    batch_size = 64  # size of training sample from reply_buffer  
    gamma = 0.70    # Discount factor  
                                       
    epsilon = 0.75  # greedy parameter # the higher, the more degree of exploration versus exploitation 
    epsilon_initial = 0.75
    epsilon_min = 0.01
    epsilon_decay = 0.005
    ddqn = True
    dueling = True
    
    #for updating the target network
    tau = 0.001 
    
    
    # Instantiate the ReplayBuffer
    replay_buffer = ReplayBuffer(max_size)
    
     # Instantiate the DQNAgent
    agent = DQNAgent(state_dim, action_dim, batch_size, gamma, epsilon, epsilon_min, epsilon_decay, 
                     epsilon_initial, tau, dueling, ddqn)
    agent.set_replay_buffer(replay_buffer)
    
       
    # Run the model
        
    metrics_data = AP(agent, replay_buffer, num_episodes, num_time_steps, Features, CVSS_df, combined_alert_data, investigation_times)
    